In [ ]:
import torch
import torch.nn as nn

In [ ]:
embed_dim = 3
hidden_dim = 2

In [ ]:
x_t = torch.tensor([[0.5, -0.2, 0.1]])
h_prev = torch.tensor([[0.1, 0.2]])
c_prev = torch.tensor([[0.0, 0.0]])

W_xi, W_hi = torch.rand((embed_dim, hidden_dim)), torch.rand((hidden_dim, hidden_dim))
W_xf, W_hf = torch.rand((embed_dim, hidden_dim)), torch.rand((hidden_dim, hidden_dim))
W_xc, W_hc = torch.rand((embed_dim, hidden_dim)), torch.rand((hidden_dim, hidden_dim))
W_xo, W_ho = torch.rand((embed_dim, hidden_dim)), torch.rand((hidden_dim, hidden_dim))

b_i = torch.rand(hidden_dim) * 0.1
b_f = torch.rand(hidden_dim) * 0.1
b_c = torch.rand(hidden_dim) * 0.1
b_o = torch.rand(hidden_dim) * 0.1

W_x = torch.cat([W_xi.T, W_xf.T, W_xc.T, W_xo.T], dim=0)
W_h = torch.cat([W_hi.T, W_hf.T, W_hc.T, W_ho.T], dim=0)
B = torch.cat([b_i.T, b_f.T, b_c.T, b_o.T], dim=0)

print(W_x.shape)

sum = x_t @ W_x.T + h_prev @ W_h.T + B  # [i, f, c, o]

i_t, f_t, c_tilde_t, o_t = torch.chunk(sum, 4, dim=1)

i_t = torch.sigmoid(i_t)
f_t = torch.sigmoid(f_t)
c_tilde_t = torch.tanh(c_tilde_t)
o_t = torch.sigmoid(o_t)


c_t = f_t * c_prev + i_t * c_tilde_t
h_t = o_t * torch.tanh(c_t)

print(c_t)
print(h_t)

tensor([[0.0690, 0.2912]])
tensor([[0.0381, 0.1699]])


In [ ]:
lstm_cell = nn.LSTMCell(embed_dim, hidden_dim)
with torch.no_grad(): # [i, f, c, o]
    lstm_cell.weight_ih.copy_([W_xi.T, W_xf.T, W_xc.T, W_xo.T], dim=0)
    lstm_cell.weight_hh.copy_([W_hi.T, W_hf.T, W_hc.T, W_ho.T], dim=0)
    lstm_cell.bias_ih.copy_(torch.cat([b_i.T, b_f.T, b_c.T, b_o.T], dim=0))
    lstm_cell.bias_hh.zero_()   
